# 🚗🚶 [인공지능 컴퓨터 비전] 픽셀 수학부터 움직임 감지 & 차량·보행자 카운터까지

### 🗺️ 이 모듈에서 배우는 것 (3줄 요약)
1. 컴퓨터의 눈에는 사진이 아니라 **숫자(픽셀)**가 보여요!
2. 이웃 숫자끼리 **뺄셈**하면 그림의 **테두리**가 보이고, **기준선(임계값)**으로 잡티를 걸러낼 수 있어요!
3. 배경과 **차이 나는 픽셀 덩어리**를 세면 자동차·보행자를 **자동으로 카운트**할 수 있어요!

### 🎒 준비물 & 선수 지식
- **준비물**: 이 노트북 + `object_detection_simulator.html` (웹캠·시뮬레이터 체험용)
- **선수 지식**: 필요 없어요! 뺄셈만 할 수 있으면 충분해요 🙂

💪 1단계부터 차근차근 따라오면, 마지막엔 여러분도 차량 카운터의 원리를 직접 설명할 수 있어요!

이 실습은 파이썬과 인공지능을 처음 배우는 학생들을 위한 맞춤형 컴퓨터 비전 수업입니다.
어려운 전문 용어 대신 **모눈종이 격자, 셀 위치(위/아래/오른쪽), 뺄셈과 절댓값, 기준선(임계값)**이라는 친숙한 수학 개념으로 컴퓨터가 이미지를 보고 움직임을 감지하는 원리를 배웁니다.

---

### 🗺️ 오늘 우리가 배울 탐구 로드맵
1. **[기초 1단계] 픽셀과 행렬 (0~255)**: 컴퓨터의 눈에는 그림이 아니라 모눈종이에 적힌 숫자가 보인다!
2. **[기초 2단계] 픽셀 뺄셈 ($|A - B|$)의 정확한 위치**:
   - **오른쪽 셀**을 빼면? $\rightarrow$ **세로선(수직 테두리)** 검출!
   - **아래쪽 셀**을 빼면? $\rightarrow$ **가로선(수평 테두리)** 검출!
   - 두 뺄셈을 합치면? $\rightarrow$ **완벽한 전체 외곽선** 완성!
3. **[기초 3단계] 임계값(Threshold) 설정**: 얼마 이상 차이 나야 진짜 테두리일까? (잡음 제거의 원리)
4. **[기초 4단계] 웹캠 & 사진 테두리 인식**: 내 얼굴과 손, 사물의 테두리를 임계값으로 직접 조절해보기!
5. **[본 실습 연결] 테두리(공간 뺄셈)에서 움직임 감지(시간 뺄셈)로!**: 배경과 차이나는 픽셀 덩어리의 크기로 오토바이와 자동차 구별하기!
6. **[본 실습 1] 고속도로 차량·오토바이 선 통과 카운터**
7. **[본 실습 2] 보행자 거리 횡단보도 인원수 카운터**


In [ ]:
#한글을 그래프에 보여주는 라이브러리
!pip install koreanize_matplotlib
import koreanize_matplotlib

## 0. 필요한 파이썬 도구 불러오기
실습에 필요한 OpenCV(컴퓨터 비전 도구), NumPy(행렬 수학 도구), Matplotlib(시각화 도구)를 준비합니다.


In [ ]:
# 🐍 파이썬 도구 불러오기: 이 셀을 가장 먼저 실행해야 아래 셀들이 동작해요!
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from IPython.display import HTML, display
import os

print("✅ 모든 파이썬 도구(OpenCV, NumPy, Matplotlib)가 준비되었습니다!")


---
## 1. [기초 1단계] 컴퓨터가 이미지를 보는 방법: 픽셀과 행렬 (0~255)

### 💡 선생님의 수학 설명
- 사람은 화면을 보면 "어? 하트네!", "자동차네!" 하고 모양으로 바로 알아챕니다.
- 하지만 **컴퓨터는 색깔이나 모양을 볼 수 없습니다.**
- 컴퓨터에게 화면은 바둑판(모눈종이)처럼 잘게 쪼개진 작은 네모 칸들로 보이며, 이 네모 칸 하나를 **픽셀(Pixel, 화소)**이라고 합니다.
- 흑백 이미지에서 각 픽셀 칸에는 **0부터 255까지의 '밝기 숫자'**만 적혀 있습니다.
  - **`0`**: 빛이 전혀 없는 가장 어두운 **검은색**
  - **`255`**: 빛이 가장 밝은 **흰색**
  - **`1 ~ 254`**: 그 사이의 점진적인 **회색 (그레이스케일)**
- 가로, 세로 격자 형태로 숫자를 늘어놓은 표를 수학에서는 **'2차원 행렬(Matrix)'**이라고 부릅니다.

✅ **체크포인트**: 255가 적힌 칸들이 모여 하트 모양이 보이나요? 컴퓨터의 눈에는 그림이 아니라 숫자 표가 보인다는 걸 확인했다면 1단계 완성! 🎉


In [ ]:
# [실습 1-1] 8x8 바둑판 격자에 숫자를 넣어 하트 모양 만들기
# 0은 검은색(바탕), 255는 흰색(하트 모양)입니다.

heart_matrix = np.array([
    [  0,   0,   0,   0,   0,   0,   0,   0],
    [  0, 255, 255,   0,   0, 255, 255,   0],
    [255, 255, 255, 255, 255, 255, 255, 255],
    [255, 255, 255, 255, 255, 255, 255, 255],
    [  0, 255, 255, 255, 255, 255, 255,   0],
    [  0,   0, 255, 255, 255, 255,   0,   0],
    [  0,   0,   0, 255, 255,   0,   0,   0],
    [  0,   0,   0,   0,   0,   0,   0,   0]
], dtype=np.uint8)

# 시각화: 좌측은 컴퓨터가 표현한 이미지, 우측은 각 칸에 적힌 실제 숫자(행렬)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# 좌측: 흑백 이미지
axes[0].imshow(heart_matrix, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("1. 사람이 보는 화면 (흑백 이미지)", fontsize=13, fontweight='bold', pad=10)
axes[0].set_xticks(range(8))
axes[0].set_yticks(range(8))
axes[0].grid(color='cyan', linestyle=':', linewidth=0.8)

# 우측: 행렬 숫자 지도 (각 칸마다 0과 255 숫자를 적어줌)
axes[1].imshow(heart_matrix, cmap='gray', vmin=0, vmax=255)
axes[1].set_title("2. 컴퓨터가 실제로 기억하는 숫자 표 (2차원 행렬)", fontsize=13, fontweight='bold', pad=10)
axes[1].set_xticks(range(8))
axes[1].set_yticks(range(8))

for r in range(8):
    for c in range(8):
        val = heart_matrix[r, c]
        txt_color = 'black' if val > 128 else '#ffd700'
        axes[1].text(c, r, str(val), ha='center', va='center', color=txt_color, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("👉 확인해보세요! 255가 적힌 칸들이 모여 하트 모양을 만들고 있습니다.")


---
## 2. [기초 2단계] 테두리(Edge) 찾기의 수학 공식: "어떤 셀을 빼야 할까?" (오른쪽 셀 vs 아래쪽 셀)

### 💡 선생님의 핵심 설명: 셀의 상대 위치 지도
컴퓨터가 테두리를 찾을 때 단순히 "이웃 셀을 뺀다"고 하면 위인지, 아래인지, 오른쪽인지 헷갈립니다.
아래처럼 **현재 셀 $(r, c)$**을 기준으로 **정확한 위치**를 정하고 뺄셈을 수행합니다:

```
                      [위쪽 셀 (r-1, c)]
                              ▲
                              │
  [왼쪽 셀 (r, c-1)]  ◀── [현재 셀 (r, c)] ──▶  [오른쪽 셀 (r, c+1)]  ──▶ 1. 가로 뺄셈 (세로선 검출!)
                              │
                              ▼
                     [아래쪽 셀 (r+1, c)]  ──▶ 2. 세로 뺄셈 (가로선 검출!)
```

---

### 🧭 뺄셈 방향에 따른 2가지 역할

#### 1) 👉 [오른쪽 셀과의 뺄셈] (가로 방향 뺄셈)
- **수학 식**:
  $$\Delta_{\text{오른쪽}}(r, c) = | \text{오른쪽 셀}(r, c+1) - \text{현재 셀}(r, c) |$$
- **원리와 역할**:
  - 왼쪽에서 오른쪽으로 이동하면서 색깔이 바뀌는 경계면을 찾습니다.
  - 따라서 **세로 방향으로 곧게 뻗은 선(수직 테두리선)**을 감지합니다!
  - 예: 하트의 왼쪽 옆구리, 오른쪽 옆구리 테두리가 선명하게 드러납니다.

#### 2) 👇 [아래쪽 셀과의 뺄셈] (세로 방향 뺄셈)
- **수학 식**:
  $$\Delta_{\text{아래쪽}}(r, c) = | \text{아래쪽 셀}(r+1, c) - \text{현재 셀}(r, c) |$$
- **원리와 역할**:
  - 위쪽에서 아래쪽으로 이동하면서 색깔이 바뀌는 경계면을 찾습니다.
  - 따라서 **가로 방향으로 누워 있는 선(수평 테두리선)**을 감지합니다!
  - 예: 하트의 윗쪽 둥근 언덕, 아래쪽 뾰족한 바닥 테두리가 선명하게 드러납니다.

#### 3) ✨ [두 뺄셈 합치기] (종합 테두리)
- **수학 식**:
  $$\Delta_{\text{종합}}(r, c) = \max(\Delta_{\text{오른쪽}}, \Delta_{\text{아래쪽}})$$
- **결과**: 가로선과 세로선이 하나로 합쳐져 **하트 모양 전체의 완전한 외곽선**이 완성됩니다!

✅ **체크포인트**: '오른쪽 셀 뺄셈은 세로선, 아래쪽 셀 뺄셈은 가로선'이라는 규칙을 말할 수 있나요? 그렇다면 2단계 완성! 🎉


In [ ]:
# [실습 1-2] 오른쪽 셀 뺄셈 vs 아래쪽 셀 뺄셈 vs 종합 테두리 비교

# 1. 오른쪽 셀과의 뺄셈: |오른쪽 셀(r, c+1) - 현재 셀(r, c)|
diff_right = np.zeros_like(heart_matrix, dtype=int)
diff_right[:, :-1] = np.abs(heart_matrix[:, 1:].astype(int) - heart_matrix[:, :-1].astype(int))

# 2. 아래쪽 셀과의 뺄셈: |아래쪽 셀(r+1, c) - 현재 셀(r, c)|
diff_down = np.zeros_like(heart_matrix, dtype=int)
diff_down[:-1, :] = np.abs(heart_matrix[1:, :].astype(int) - heart_matrix[:-1, :].astype(int))

# 3. 두 뺄셈의 종합 (둘 중 큰 차이값 선택)
diff_total = np.maximum(diff_right, diff_down)

# 4개 화면을 나란히 시각화
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# (1) 원본 하트
axes[0].imshow(heart_matrix, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("1. 원본 하트 (8x8)", fontsize=12, fontweight='bold')

# (2) 오른쪽 셀과의 뺄셈
axes[1].imshow(diff_right, cmap='hot', vmin=0, vmax=255)
axes[1].set_title("2. 오른쪽 셀 뺄셈 (👉 세로선 검출)", fontsize=12, color='cyan', fontweight='bold')

# (3) 아래쪽 셀과의 뺄셈
axes[2].imshow(diff_down, cmap='hot', vmin=0, vmax=255)
axes[2].set_title("3. 아래쪽 셀 뺄셈 (👇 가로선 검출)", fontsize=12, color='orange', fontweight='bold')

# (4) 종합 테두리
axes[3].imshow(diff_total, cmap='hot', vmin=0, vmax=255)
axes[3].set_title("4. 종합 테두리 (👉 + 👇 완성)", fontsize=12, color='lime', fontweight='bold')

# 각 칸에 숫자 표기
for idx, data in enumerate([heart_matrix, diff_right, diff_down, diff_total]):
    ax = axes[idx]
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    for r in range(8):
        for c in range(8):
            v = data[r, c]
            txt_c = 'black' if v > 128 else 'white'
            ax.text(c, r, str(v), ha='center', va='center', color=txt_c, fontsize=7, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 배운 점 확인:")
print(f" • 오른쪽 셀을 뺐을 때 (|오른쪽 - 현재|): {np.count_nonzero(diff_right)}개의 세로 경계선 발견!")
print(f" • 아래쪽 셀을 뺐을 때 (|아래쪽 - 현재|): {np.count_nonzero(diff_down)}개의 가로 경계선 발견!")
print(f" • 둘을 합친 종합 테두리: {np.count_nonzero(diff_total)}개의 완전한 외곽선 완성!")


In [ ]:
# [실습 1-2 심화] 내가 그린 그림 JPG를 업로드해서 선 검출하기
# 직접 그린 그림이나 고문서 사진을 올려보세요!

from google.colab import files
import cv2

# 1. JPG 파일 업로드
print("📤 JPG 파일을 선택해주세요!")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"✅ 업로드 완료: {filename}")

# 2. 흑백으로 불러오기
img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
print(f"   사진 크기: {img.shape[1]} × {img.shape[0]} 픽셀")

# 3. 오른쪽 셀 뺄셈 (세로선 검출)
my_right = np.zeros_like(img, dtype=int)
my_right[:, :-1] = np.abs(img[:, 1:].astype(int) - img[:, :-1].astype(int))

# 4. 아래쪽 셀 뺄셈 (가로선 검출)
my_down = np.zeros_like(img, dtype=int)
my_down[:-1, :] = np.abs(img[1:, :].astype(int) - img[:-1, :].astype(int))

# 5. 종합 테두리
my_total = np.maximum(my_right, my_down)

# 6. 나란히 보여주기
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title("1. 내가 올린 그림", fontsize=12, fontweight='bold')
axes[0].axis('off')
axes[1].imshow(my_right, cmap='hot')
axes[1].set_title("2. 세로선 검출 👉", fontsize=12, color='cyan', fontweight='bold')
axes[1].axis('off')
axes[2].imshow(my_down, cmap='hot')
axes[2].set_title("3. 가로선 검출 👇", fontsize=12, color='orange', fontweight='bold')
axes[2].axis('off')
axes[3].imshow(my_total, cmap='hot')
axes[3].set_title("4. 종합 테두리 ✨", fontsize=12, color='lime', fontweight='bold')
axes[3].axis('off')
plt.tight_layout()
plt.show()

print("💡 내 그림에서도 가로선·세로선이 잘 검출되나요?")

---
## 3. [기초 3단계] 임계값(Threshold)의 원리: 기준선 정하기 (약한 선 vs 또렷한 선)

### 💡 선생님의 수학 설명
- 실제 사진이나 웹캠 카메라 화면에는 조명이나 그림자, 먼지 때문에 차이가 **15, 30, 45** 정도로 살짝 나는 부분이 무수히 많습니다.
- 이것들을 전부 테두리(선)로 그리면 화면에 모래알 같은 지저분한 잡티(노이즈)가 가득 차게 됩니다.
- 그래서 컴퓨터에게 **"기준선(임계값, Threshold)"**을 정해줍니다:
  $$\text{테두리 픽셀} = \begin{cases} 255 \text{ (진짜 선으로 인정)}, & \text{if } \text{차이값} \ge \text{Threshold} \\ 0 \text{ (버리기, 검은색)}, & \text{if } \text{차이값} < \text{Threshold} \end{cases}$$
- **임계값 조절의 효과**:
  - **임계값을 너무 낮추면 (`Threshold = 25`)**: 바람에 흔들리는 작은 먼지나 미세한 그림자까지 다 선으로 그려져 화면이 지저분해집니다.
  - **임계값이 적당하면 (`Threshold = 100`)**: 물체의 또렷한 외곽선만 깔끔하게 남습니다.
  - **임계값을 너무 높이면 (`Threshold = 180`)**: 진짜 테두리 선마저 끊어져서 사라집니다.

✅ **체크포인트**: 임계값을 낮추면 잡티가 많아지고, 높이면 선이 끊긴다는 걸 확인했나요? 그렇다면 3단계 완성! 🎉


In [ ]:
# [실습 1-3] 그림자와 노이즈가 섞인 이미지에서 임계값(Threshold) 변경 실험

# 그림자(서서히 밝아지는 그라데이션)와 노이즈가 섞인 10x10 이미지 생성
np.random.seed(42)
noisy_image = np.zeros((10, 10), dtype=np.float32)
# 가운데 4x4 사각형 물체 (밝기 200)
noisy_image[3:7, 3:7] = 200
# 왼쪽에서 오른쪽으로 은은한 그림자 (밝기 10~50)
noisy_image += np.tile(np.linspace(10, 50, 10), (10, 1))
# 카메라 센서의 자잘한 노이즈 (랜덤 값 0~25)
noisy_image += np.random.uniform(0, 25, (10, 10))
noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)

# 1. 오른쪽 셀 뺄셈
d_r = np.zeros_like(noisy_image, dtype=int)
d_r[:, :-1] = np.abs(noisy_image[:, 1:].astype(int) - noisy_image[:, :-1].astype(int))

# 2. 아래쪽 셀 뺄셈
d_d = np.zeros_like(noisy_image, dtype=int)
d_d[:-1, :] = np.abs(noisy_image[1:, :].astype(int) - noisy_image[:-1, :].astype(int))

# 종합 차이값 행렬
diff_noisy = np.maximum(d_r, d_d)

# 3가지 임계값으로 비교해보기
t_low = 25    # 기준이 너무 낮음 (잡음까지 선으로 인식)
t_mid = 100   # 적당한 기준 (사각형 테두리만 깔끔하게)
t_high = 180  # 기준이 너무 높음 (선이 끊어짐)

edge_low = np.where(diff_noisy >= t_low, 255, 0).astype(np.uint8)
edge_mid = np.where(diff_noisy >= t_mid, 255, 0).astype(np.uint8)
edge_high = np.where(diff_noisy >= t_high, 255, 0).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(noisy_image, cmap='gray')
axes[0].set_title("1. 원본 (그림자+노이즈 포함)", fontsize=11, fontweight='bold')

axes[1].imshow(edge_low, cmap='gray')
axes[1].set_title(f"2. 임계값={t_low} (너무 낮음: 잡티 가득)", fontsize=11, color='red', fontweight='bold')

axes[2].imshow(edge_mid, cmap='gray')
axes[2].set_title(f"3. 임계값={t_mid} (적당함: 선명한 테두리)", fontsize=11, color='green', fontweight='bold')

axes[3].imshow(edge_high, cmap='gray')
axes[3].set_title(f"4. 임계값={t_high} (너무 높음: 선 끊김)", fontsize=11, color='blue', fontweight='bold')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

print("🔬 학생 탐구 과제: t_mid 값을 60, 80, 120으로 바꿔보면서 가장 깨끗한 테두리를 찾아보세요!")


---
## 4. [기초 4단계] 실제 사진 & 웹캠에서 테두리 인식 (Canny 알고리즘과 임계값)

### 💡 컴퓨터 비전의 명작: Canny 에지 검출기
- 실제 자율주행차나 스마트폰 카메라에서는 우리가 방금 배운 **가로(오른쪽 셀)·세로(아래쪽 셀) 뺄셈과 임계값**을 고속으로 계산해주는 **Canny 알고리즘 (`cv2.Canny`)**을 주로 사용합니다.
- Canny 알고리즘에는 2개의 임계값 기준이 있습니다:
  - **`THRESHOLD_LOW` (하한 기준)**: 이 기준보다 낮으면 무조건 쓰레기(잡음)로 버립니다.
  - **`THRESHOLD_HIGH` (상한 기준)**: 이 기준보다 높으면 무조건 확실한 테두리로 인정합니다.
  - 그 사이의 값은 확실한 테두리와 이어져 있을 때만 선으로 인정해줍니다.

✅ **체크포인트**: Canny의 두 기준(낮으면 버리기, 높으면 인정하기)을 설명할 수 있나요? 그렇다면 4단계 완성! 🎉


In [ ]:
# [실습 1-4A] 모의 도로 차량 사진에서 임계값을 바꿔가며 테두리 찾아보기

# 실습용 모의 차량 실루엣 이미지 자동 생성
sample_car = np.ones((260, 420), dtype=np.uint8) * 60 # 아스팔트 배경

# 차체 그리기 (차체, 창문, 바퀴)
cv2.rectangle(sample_car, (70, 130), (350, 200), 220, -1)     # 차체 아래
cv2.rectangle(sample_car, (120, 75), (300, 130), 200, -1)     # 차체 위 지붕
cv2.rectangle(sample_car, (135, 85), (200, 125), 100, -1)     # 앞 유리창
cv2.rectangle(sample_car, (215, 85), (285, 125), 100, -1)     # 뒤 유리창
cv2.circle(sample_car, (125, 200), 28, 20, -1)                 # 앞바퀴
cv2.circle(sample_car, (295, 200), 28, 20, -1)                 # 뒷바퀴
cv2.circle(sample_car, (125, 200), 12, 180, -1)                # 앞바퀴 휠
cv2.circle(sample_car, (295, 200), 12, 180, -1)                # 뒷바퀴 휠

# ========================================================
# 🔧 [학생 실습 변수] 임계값 숫자를 바꿔보세요!
# ========================================================
THRESH_LOW  = 50    # 하한 임계값 (기본: 50)
THRESH_HIGH = 150   # 상한 임계값 (기본: 150)

# Canny 테두리 검출 실행
car_edges = cv2.Canny(sample_car, THRESH_LOW, THRESH_HIGH)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sample_car, cmap='gray')
axes[0].set_title("1. 원본 자동차 이미지 (도로 위)", fontsize=13, fontweight='bold', pad=10)
axes[0].axis('off')

axes[1].imshow(car_edges, cmap='gray')
axes[1].set_title(f"2. 뺄셈과 임계값({THRESH_LOW}, {THRESH_HIGH})으로 찾은 외곽선", fontsize=13, color='cyan', fontweight='bold', pad=10)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"✅ 외곽선 추출 완료! 감지된 테두리 픽셀 수: {np.count_nonzero(car_edges)}개")


### 📹 (선택 실습) 내 웹캠으로 직접 테두리 찍어보기 & 웹 시뮬레이터 안내
- **방법 1 (웹 브라우저 시뮬레이터)**: 우리가 함께 사용하는 **`object_detection_simulator.html`**의 **2번째 탭 [웹캠 테두리 인식]**을 열면, 웹캠 화면을 보면서 슬라이더로 30FPS 실시간 테두리 임계값을 즉시 조절할 수 있습니다!
- **방법 2 (구글 코랩 웹캠 캡처)**: 아래 셀(방법 2)을 실행하면 웹캠 사진 촬영 버튼이 나타납니다. 내 손바닥이나 얼굴을 찰칵 찍어 테두리가 어떻게 나오는지 바로 확인해보세요!
- **방법 3 (Gradio 웹캠 앱)**: 그 다음 셀(방법 3)을 실행하면 Gradio 웹캠 앱이 실행됩니다. 코랩이 아닌 로컬 주피터에서도 동작합니다!

In [ ]:
# [실습 1-4B] 구글 코랩 웹캠으로 내 손/얼굴 찰칵 찍어서 테두리 확인하기
try:
    from google.colab.output import eval_js
    from IPython.display import Javascript
    from base64 import b64decode
    
    def take_photo(filename='my_photo.jpg'):
        js = Javascript('''
        async function takePhoto() {
            const div = document.createElement("div");
            const btn = document.createElement("button");
            btn.textContent = "📸 웹캠 사진 찰칵 찍기";
            btn.style.cssText = "padding:10px 20px; background:#03c75a; color:#fff; border:none; border-radius:6px; font-weight:bold; cursor:pointer;";
            div.appendChild(btn);

            const video = document.createElement("video");
            video.style.cssText = "display:block; margin-top:10px; border-radius:8px; max-width:400px;";
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            await new Promise((resolve) => btn.onclick = resolve);
            const canvas = document.createElement("canvas");
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext("2d").drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL("image/jpeg", 0.85);
        }
        ''')
        display(js)
        data = eval_js("takePhoto()")
        binary = b64decode(data.split(",")[1])
        with open(filename, "wb") as f:
            f.write(binary)
        return filename

    print("📸 웹캠이 준비되었습니다. 사진을 찍으려면 아래 함수를 실행하세요:")
    print("photo_file = take_photo()")
except Exception as e:
    print("ℹ️ 로컬 주피터 환경이거나 웹캠 지원 브라우저가 아닐 경우, HTML 시뮬레이터(object_detection_simulator.html)의 2번째 탭을 이용해주세요!")


In [ ]:
# [실습 1-4B] Gradio 버전: 웹캠으로 내 손/얼굴 찰칵 찍어서 테두리 확인하기
# 코랩의 eval_js 대신 Gradio를 사용합니다. 로컬 주피터에서도 동작합니다.
try:
    import gradio as gr
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
    import gradio as gr

import numpy as np
import cv2
import matplotlib.pyplot as plt

# 1. 웹캠에서 사진 찍기
webcam = gr.Image(sources=["webcam"], type="numpy", label="📸 웹캠: 촬영 버튼을 눌러주세요")

# 2. 촬영된 이미지를 변수에 저장
img = webcam.value if hasattr(webcam, 'value') else None

def show_edges(img):
    # 3. 그레이스케일로 바꾸기
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # 4. Canny 테두리 찾기
    edges = cv2.Canny(gray, 50, 150)
    # 5. 맷플롯립으로 차례대로 그리기
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img)
    axes[0].set_title("원본 사진")
    axes[0].axis("off")
    axes[1].imshow(edges, cmap="gray")
    axes[1].set_title("찾은 테두리")
    axes[1].axis("off")
    plt.tight_layout()
    return fig

with gr.Blocks(title="실습 1-4B") as demo:
    gr.Markdown("## 📸 [실습 1-4B] 웹캠으로 테두리 확인하기")
    inp = gr.Image(sources=["webcam"], type="numpy", label="웹캠 촬영")
    btn = gr.Button("📸 찰칵! 테두리 보기", variant="primary")
    out = gr.Plot(label="결과")
    btn.click(fn=show_edges, inputs=inp, outputs=out)

demo.launch(share=False)


---
## 5. [핵심 연결고리] 테두리(공간 뺄셈)에서 움직임 감지(시간 뺄셈)와 물체 크기 구별로!

### 🌉 지금까지 배운 수학이 어떻게 자율주행과 차량 인식으로 이어질까요?
선생님과 함께 지금까지 배운 3가지 개념을 하나의 흐름으로 꿰뚫어 봅시다!

| 단계 | 테두리 찾기 (사진 1장) | 움직이는 물체 감지 (연속 동영상) |
|---|---|---|
| **1. 뺄셈 대상** | **공간의 차이**: $| \text{오른쪽/아래쪽 픽셀} - \text{현재 픽셀} |$ | **시간의 차이**: $| \text{현재 프레임 픽셀} - \text{배경 도로 픽셀} |$ |
| **2. 임계값(Threshold)** | 차이가 임계값 이상이면 $\rightarrow$ **테두리 선** | 차이가 임계값 이상이면 $\rightarrow$ **움직이는 물체 영역(흰색 덩어리)** |
| **3. 크기 필터(`MIN_AREA`)** | (선 두께) | 흰색 덩어리의 **픽셀 개수(면적)**를 세어서 **오토바이/보행자 vs 승용차** 구별! |

- **배경 차분(Background Subtraction)**: 멈춰 있는 배경 화면을 미리 찍어두고, 현재 화면에서 빼는 방법이에요. 차이가 나는 부분 = 움직인 물체!

- **물체 크기(`MIN_AREA`)의 비밀**:
  - 오토바이나 보행자는 화면에서 차지하는 흰색 픽셀 수가 작습니다 (약 100~300개).
  - 큰 승용차나 화물 트럭은 흰색 픽셀 수가 매우 큽니다 (약 450~2000개 이상).
  - 따라서 **"최소 크기 기준(`MIN_AREA`)"을 100으로 낮추면 오토바이까지 모두 감지**하고,
  - **450으로 높이면 작은 오토바이는 쏙 걸러내고 큰 자동차만 골라서 세게** 됩니다!

✅ **체크포인트**: '공간 뺄셈(테두리) → 시간 뺄셈(움직임) → 크기 필터(구별)' 흐름을 설명할 수 있나요? 그렇다면 핵심 연결고리 완성! 🎉


---
## 6. [본 실습 1] 고속도로 차량 & 오토바이 주행 영상 준비
학생들이 다운로드할 필요 없이, **오토바이(작은 물체 3대) + 승용차(중간 물체 8대) + 대형트럭(큰 물체 4대)**가 섞여서 달리는 모의 고속도로 영상을 6초 만에 생성합니다.


In [ ]:
# 실습용 모의 도로 영상 자동 생성 (오토바이 + 승용차 + 대형트럭 주행)
video_path = "car_highway2.mp4"

vw, vh, vfps = 640, 360, 30
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
temp_out = cv2.VideoWriter(video_path, fourcc, vfps, (vw, vh))

np.random.seed(2024)
lanes = [120, 240, 360, 480]
sim_objects = []

# 15개의 다양한 크기 물체 생성 (오토바이, 승용차, 트럭)
for i in range(15):
    obj_type = np.random.choice(['motorcycle', 'car', 'truck'], p=[0.4, 0.4, 0.2])
    if obj_type == 'motorcycle':
        w, h = 18, 38  # 오토바이 (작은 크기: 면적 약 680)
        color = (255, 140, 0)
        speed = float(np.random.uniform(5.5, 7.5))
    elif obj_type == 'car':
        w, h = 38, 70  # 승용차 (중간 크기: 면적 약 2600)
        color = (0, 200, 255)
        speed = float(np.random.uniform(4.0, 5.8))
    else:
        w, h = 52, 110 # 대형 트럭 (큰 크기: 면적 약 5700)
        color = (120, 120, 240)
        speed = float(np.random.uniform(3.0, 4.2))
        
    lane_x = int(np.random.choice(lanes) + np.random.randint(-15, 15))
    start_y = -int(np.random.randint(50, 800))
    sim_objects.append({
        'type': obj_type, 'w': w, 'h': h, 'color': color,
        'speed': speed, 'x': lane_x, 'y': float(start_y)
    })

# 180프레임 (6초) 동안 도로 시뮬레이션 영상 녹화
for f_idx in range(vfps * 6):
    frame = np.ones((vh, vw, 3), dtype=np.uint8) * 55 # 아스팔트 회색 도로
    
    # 도로 차선 (흰색 점선)
    for lx in [180, 300, 420]:
        for dy in range(0, vh, 40):
            dash_y = (dy + f_idx * 4) % vh
            cv2.line(frame, (lx, dash_y), (lx, min(dash_y + 20, vh)), (220, 220, 220), 2)
            
    # 노변 가드레일
    cv2.line(frame, (60, 0), (60, vh), (180, 180, 180), 3)
    cv2.line(frame, (vw - 60, 0), (vw - 60, vh), (180, 180, 180), 3)
    
    # 물체들 이동
    for obj in sim_objects:
        obj['y'] += obj['speed']
        cy = int(obj['y'])
        cx = int(obj['x'])
        if -150 < cy < vh + 150:
            x1, y1 = cx - obj['w'] // 2, cy - obj['h'] // 2
            x2, y2 = cx + obj['w'] // 2, cy + obj['h'] // 2
            
            # 그림자
            cv2.ellipse(frame, (cx, cy + 4), (obj['w'] // 2 + 4, obj['h'] // 2 + 4), 0, 0, 360, (30, 30, 30), -1)
            # 차체 본체
            cv2.rectangle(frame, (x1, y1), (x2, y2), obj['color'], -1)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (20, 20, 20), 1)
            
            # 헤드라이트 / 후미등
            cv2.circle(frame, (x1 + 4, y2 - 4), 3, (0, 0, 255), -1)
            cv2.circle(frame, (x2 - 4, y2 - 4), 3, (0, 0, 255), -1)

    temp_out.write(frame)

temp_out.release()
print(f"🎬 모의 고속도로 영상({video_path}) 생성 완료! (15대 주행: 오토바이/승용차/트럭)")


---
## 7. [학생 실험] 물체 크기 조절하기: 오토바이 포함 vs 자동차만 인식

### 🎛️ 학생들이 직접 실험해보는 2가지 설정:
- **실험 1: `MIN_AREA = 100` (작게)** $\rightarrow$ **오토바이까지 모두 감지! (초록 박스 생성 & 카운트 증가)**
- **실험 2: `MIN_AREA = 450` (크게)** $\rightarrow$ 작은 오토바이는 무시하고 **큰 승용차·트럭만 골라서 감지!**

✅ **체크포인트**: `MIN_AREA`를 100으로 했을 때와 450으로 했을 때 카운트가 어떻게 달라지나요? 직접 바꿔보고 아래 기록표에 적어보세요! 🎉


In [ ]:
# ========================================================
# 🔧 [학생 실습 변수] 숫자를 직접 바꿔보세요!
# ========================================================
MIN_AREA    = 100    # ⭐ [핵심!] 감지할 최소 크기 (100: 오토바이 포함 / 450: 자동차만)
LINE_RATIO  = 0.60   # 노란색 통과선 높이 (0.0=맨위, 1.0=맨아래)
DILATE_ITER = 2      # 조각난 흰색 점 뭉쳐주기 강도

# 카메라 탐색 구역 (위 / 아래 범위)
ROI_TOP     = 0.20
ROI_BOTTOM  = 0.85

# ========================================================
# 분석 실행
# ========================================================
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_path = "experiment_result.mp4"
out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W * 2, H))

# 움직임 감지기 (시간적 차이값 계산: 배경 차분 MOG2)
# MOG2는 배경 화면을 스스로 학습하는 OpenCV의 움직임 감지 도구예요.
bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=50, detectShadows=True)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
line_y = int(H * LINE_RATIO)

roi_mask = np.zeros((H, W), dtype=np.uint8)
roi_y1, roi_y2 = int(H * ROI_TOP), int(H * ROI_BOTTOM)
cv2.rectangle(roi_mask, (40, roi_y1), (W - 40, roi_y2), 255, -1)

# 꼬리표(ID) 추적기
next_id = 1
# 🚗 지나가는 물체마다 번호표(ID)를 붙여 기억하는 '출석부'예요
tracked_objects = {} # { id: {'centroid': (cx, cy), 'prev_y': py, 'counted': False} }
TOTAL_COUNT = 0

print(f"🚀 물체 인식 시작! (설정된 최소 크기: {MIN_AREA} 픽셀)")
if MIN_AREA <= 200:
    print("🛵 [상태] 작은 오토바이까지 모두 감지하도록 설정되었습니다.")
else:
    print("🚗 [상태] 오토바이는 무시하고 큰 자동차/트럭만 감지하도록 설정되었습니다.")

while True:
    ok, frame = cap.read()
    if not ok: break
    
    # 1. 흑백 움직임 화면 만들기 (배경과 차이나는 픽셀 뺄셈)
    fg_mask = bg_subtractor.apply(frame)
    fg_mask = cv2.bitwise_and(fg_mask, fg_mask, mask=roi_mask)
    _, fg_thresh = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)
    fg_clean = cv2.morphologyEx(fg_thresh, cv2.MORPH_OPEN, kernel)
    fg_clean = cv2.dilate(fg_clean, kernel, iterations=DILATE_ITER)
    
    # 2. 흰색 덩어리들의 윤곽선 찾기
    contours, _ = cv2.findContours(fg_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    current_centroids = []
    current_boxes = []
    
    for c in contours:
        area = cv2.contourArea(c)
        # ⭐ [수학적 크기 필터] 학생이 설정한 크기(MIN_AREA)보다 큰 덩어리만 물체로 인정!
        if area >= MIN_AREA:
            bx, by, bw, bh = cv2.boundingRect(c)
            cx = bx + bw // 2
            cy = by + bh // 2
            current_centroids.append((cx, cy))
            current_boxes.append((bx, by, bw, bh))
            
    # 3. 꼬리표(ID) 매칭 및 노란선 통과 카운팅
    matched_ids = set()
    for (cx, cy) in current_centroids:
        matched = False
        for oid, data in tracked_objects.items():
            ox, oy = data['centroid']
            if np.hypot(cx - ox, cy - oy) < 60:
                matched_ids.add(oid)
                prev_y = data['centroid'][1]
                data['prev_y'] = prev_y
                data['centroid'] = (cx, cy)
                matched = True
                
                # 선 통과 체크
                if not data['counted']:
                    if (prev_y < line_y and cy >= line_y) or (abs(cy - line_y) <= 8):
                        TOTAL_COUNT += 1
                        data['counted'] = True
                break
        if not matched:
            tracked_objects[next_id] = {'centroid': (cx, cy), 'prev_y': cy, 'counted': False}
            next_id += 1
            
    # 4. 화면에 그리기
    # 노란색 통과선
    cv2.line(frame, (0, line_y), (W, line_y), (0, 255, 255), 2)
    cv2.putText(frame, f"Count Line (Y={line_y})", (15, line_y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    # 초록색 감지 상자
    for (bx, by, bw, bh) in current_boxes:
        cv2.rectangle(frame, (bx, by), (bx + bw, by + bh), (0, 255, 0), 2)
        cv2.circle(frame, (bx + bw // 2, by + bh // 2), 4, (0, 0, 255), -1)
        
    # 카운트 전광판 표시
    cv2.rectangle(frame, (10, 10), (240, 50), (0, 0, 0), -1)
    cv2.putText(frame, f"TOTAL COUNT: {TOTAL_COUNT}", (20, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 128), 2)
    
    # 5. 좌(사람 눈) / 우(컴퓨터 눈 흑백 마스크) 듀얼 뷰 영상 저장
    mask_bgr = cv2.cvtColor(fg_clean, cv2.COLOR_GRAY2BGR)
    cv2.line(mask_bgr, (0, line_y), (W, line_y), (0, 255, 255), 2)
    cv2.putText(mask_bgr, "Computer Eye (MOG2 Mask)", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    
    dual_frame = np.hstack((frame, mask_bgr))
    out.write(dual_frame)

cap.release()
out.release()

print(f"🎉 분석 완료! 최종 통과 개수: {TOTAL_COUNT}대")


---
## 8. [본 실습 2] 사람들의 이동(보행자 거리 횡단보도) 영상으로 실습하기
횡단보도를 건너는 사람들을 감지하여 통과한 사람 수를 카운트하는 모드입니다. 사람 크기에 맞게 `MIN_AREA = 80` 정도로 설정합니다.

✅ **체크포인트**: 차량 영상과 같은 코드 구조로 보행자도 셀 수 있다는 걸 확인했나요? 숫자 하나(`MIN_AREA`)만 바꾸면 새로운 문제에 응용할 수 있어요! 🎉


In [ ]:
# 보행자 횡단보도 영상 생성
# 📌 위 6번 셀에서 만든 변수(fourcc, vfps, vw, vh)를 그대로 재사용합니다. 6번 셀을 먼저 실행해주세요!
ped_path = "pedestrian_walk.mp4"
temp_ped = cv2.VideoWriter(ped_path, fourcc, vfps, (vw, vh))

people = []
for i in range(12):
    people.append({
        'x': np.random.randint(80, vw - 80),
        'y': -np.random.randint(20, 400),
        'speed': np.random.uniform(1.8, 2.8),
        'color': (int(np.random.randint(80, 255)), int(np.random.randint(80, 255)), int(np.random.randint(80, 255)))
    })

for f_idx in range(vfps * 6):
    frame = np.ones((vh, vw, 3), dtype=np.uint8) * 80 # 보도블록
    
    # 횡단보도 흰색 줄무늬
    for wx in range(100, vw - 100, 50):
        cv2.rectangle(frame, (wx, 120), (wx + 30, 240), (220, 220, 220), -1)
        
    for p in people:
        p['y'] += p['speed']
        py = int(p['y'])
        px = int(p['x'])
        if -50 < py < vh + 50:
            # 보행자 머리 + 몸통
            cv2.circle(frame, (px, py - 12), 7, (200, 180, 150), -1)
            cv2.rectangle(frame, (px - 8, py - 5), (px + 8, py + 18), p['color'], -1)
            
    temp_ped.write(frame)

temp_ped.release()
print(f"🚶 보행자 횡단보도 영상({ped_path}) 생성 완료! (12명 보행)")


---
## 9. 분석 결과 미리보기 (스틸컷)
좌측(사람이 보는 화면)과 우측(컴퓨터가 보는 흑백 마스크 화면)을 나란히 확인합니다.

✅ **체크포인트**: 왼쪽(사람의 눈)과 오른쪽(컴퓨터의 눈)이 어떻게 다른지 친구에게 설명해보세요! 🎉


In [ ]:
cap_check = cv2.VideoCapture("experiment_result.mp4")
cap_check.set(cv2.CAP_PROP_POS_FRAMES, 60)
ret, sample = cap_check.read()
cap_check.release()

if ret:
    cv2.imwrite("result_view.jpg", sample)
    print("📷 [분석 결과 미리보기] 좌측(사람의 눈) vs 우측(컴퓨터의 눈 흑백 마스크)")
    display(HTML('<img src="result_view.jpg" width="850" style="border-radius:6px; border:1px solid #ccc;"/>'))
else:
    print("결과 프레임을 불러오지 못했습니다.")


---
## 🔬 학생 탐구 실험 표 & 퀴즈

### 1. 내가 실험한 결과 기록표

| 실험 번호 | 내가 설정한 크기 (`MIN_AREA`) | 오토바이 인식 여부 | 자동차 인식 여부 | 결과 카운트 | 배운 점 |
|---|---|---|---|---|---|
| **실험 1** | `MIN_AREA = 100` (작게) | 감지됨 (초록 네모) | 감지됨 (초록 네모) | 많음 | 작은 오토바이와 사람도 감지할 수 있다 |
| **실험 2** | `MIN_AREA = 450` (크게) | **무시됨 (네모 안 뜸)** | 감지됨 (초록 네모) | 적음 | 작은 물체를 걸러내고 큰 자동차만 셀 수 있다 |

---

### 2. 오늘의 퀴즈 (복습)
1. **컴퓨터가 이미지를 볼 때, 가장 어두운 검은색은 숫자 몇이고 가장 밝은 흰색은 숫자 몇인가요?**
   - *정답: 검은색은 0, 흰색은 255*
2. **테두리를 찾을 때 '오른쪽 셀'을 빼면 어떤 선이 발견되고, '아래쪽 셀'을 빼면 어떤 선이 발견되나요?**
   - *정답: 오른쪽 셀 뺄셈은 세로선(수직선), 아래쪽 셀 뺄셈은 가로선(수평선)*
3. **미세한 먼지나 그림자를 걸러내기 위해 정해주는 '기준 점수'를 무엇이라고 부르나요?**
   - *정답: 임계값 (Threshold)*
4. **동영상에서 움직이는 자동차를 찾을 때는 어떤 두 화면을 뺄셈하나요?**
   - *정답: 현재 프레임 화면과 멈춰있는 배경 화면*
5. **오토바이는 무시하고 큰 트럭과 승용차만 골라서 세려면 `MIN_AREA` 숫자를 어떻게 해야 할까요?**
   - *정답: 숫자를 450 이상으로 크게 높인다.*
